# Sequence Construction for sequential models
In this notebook, the purpose is to create equal length sequences for sequential models. The preparation includes scaling based on the training set, truncation of long sequences, zero-padding of shorter sequences, construction of binary masks, and preservation of inter-readout time gaps to support modelling of irregular temporal sampling.

In [1]:
# import required libraries
import pickle
from pathlib import Path

import numpy as np
import pandas as pd


In [2]:
# import paths
import sys
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
SEQUENCE_DIR = PROJECT_ROOT / "data" / "processed" / "sequences"
SEQUENCE_DIR.mkdir(parents=True, exist_ok=True)

# train_raw_sequence_path = SEQUENCE_DIR / "train_seq_full.pkl"
# val_raw_sequence_path = SEQUENCE_DIR / "val_seq_full.pkl"
# test_raw_sequence_path = SEQUENCE_DIR / "test_seq_full.pkl"

In [3]:
# raw truncated sequences (unnormalized)
train_trunc_raw_sequence_path = SEQUENCE_DIR / "train_seq_truncated.pkl"
val_trunc_raw_sequence_path = SEQUENCE_DIR / "val_seq_truncated.pkl"
test_trunc_raw_sequence_path = SEQUENCE_DIR / "test_seq_truncated.pkl"


In [24]:
# load the train raw sequence dictionary
with open(train_raw_sequence_path, "rb") as f:
    train_seq = pickle.load(f)

print(f"Number of vehicles: {len(train_seq)}")

Number of vehicles: 16485


In [4]:
# load the truncated train raw sequence dictionary
with open(train_trunc_raw_sequence_path, "rb") as f:
    train_trunc_seq = pickle.load(f)

print(f"Number of vehicles: {len(train_trunc_seq)}")

Number of vehicles: 16485


In [25]:
# load the val raw sequence dictionary
with open(val_raw_sequence_path, "rb") as f:
    val_seq = pickle.load(f)

print(f"Number of vehicles: {len(val_seq)}")

Number of vehicles: 3532


In [5]:
# load the truncated val raw sequence dictionary
with open(val_trunc_raw_sequence_path, "rb") as f:
    val_trunc_seq = pickle.load(f)

print(f"Number of vehicles: {len(val_trunc_seq)}")

Number of vehicles: 3532


In [26]:
# load the test raw sequence dictionary
with open(test_raw_sequence_path, "rb") as f:
    test_seq = pickle.load(f)

print(f"Number of vehicles: {len(test_seq)}")

Number of vehicles: 3533


In [6]:
# load the truncated test raw sequence dictionary
with open(test_trunc_raw_sequence_path, "rb") as f:
    test_trunc_seq = pickle.load(f)

print(f"Number of vehicles: {len(test_trunc_seq)}")

Number of vehicles: 3533


In [9]:
# # convert sequences to a list for easier handling
# train_seq_list = list(train_seq.values())
# val_seq_list = list(val_seq.values())
# test_seq_list = list(test_seq.values())
# print(type(train_seq_list), len(train_seq_list))

<class 'list'> 16485


In [7]:
# Impute (forward fill + zero fill) and normalize sequences 
# fit normalizer on training sequences, apply normalization to all splits
# histogram features transformed with log transformation to preserve monotonicity,
# and skewness
import importlib
import src.sequence_builder as sequence_builder
importlib.reload(sequence_builder)
# from src.sequence_builder import preprocess_sequence_splits
# train_seq_norm, val_seq_norm, test_seq_norm, seq_normalizer = sequence_builder.preprocess_sequence_splits(
#     train_seq=train_seq, 
#     val_seq=val_seq,
#     test_seq=test_seq
# )
# from src.sequence_builder import preprocess_sequence_splits
train_trunc_seq_norm, val_trunc_seq_norm, test_trunc_seq_norm, seq_trunc_normalizer = sequence_builder.preprocess_sequence_splits(
    train_seq=train_trunc_seq, 
    val_seq=val_trunc_seq,
    test_seq=test_trunc_seq
)

In [29]:
train_imp = sequence_builder.impute_sequence_dict(train_seq)

for vid, seq in train_imp.items():
    x = np.asarray(seq["numerical_sequence"], dtype=np.float64)
    if np.isnan(x).any():
        print("Still has NaNs:", vid)
        break
else:
    print("All training sequences imputed successfully.")

All training sequences imputed successfully.


In [8]:
# check imputed truncated sequences
train_trunc_imp = sequence_builder.impute_sequence_dict(train_trunc_seq)

for vid, seq in train_trunc_imp.items():
    x = np.asarray(seq["numerical_sequence"], dtype=np.float64)
    if np.isnan(x).any():
        print("Still has NaNs:", vid)
        break
else:
    print("All training sequences imputed successfully.")

All training sequences imputed successfully.


In [30]:
# verify if the imputation, normalization and log transformation have worked
print(len(train_seq_norm), len(val_seq_norm), len(test_seq_norm))
print(seq_normalizer.keys())

first_vehicle = next(iter(train_seq_norm))
print(train_seq_norm[first_vehicle].keys())
print(train_seq_norm[first_vehicle]["numerical_sequence"].shape)
print(train_seq_norm[first_vehicle]["histogram_sequence"].shape)

16485 3532 3533
dict_keys(['means', 'stds'])
dict_keys(['vehicle_id', 'time_steps', 'time_gaps', 'numerical_sequence', 'histogram_sequence', 'sequence_length', 'static_features', 'duration', 'event'])
(80, 8)
(80, 97)


In [10]:
# verify if the imputation, normalization and log transformation have worked for truncated sequences
print(len(train_trunc_seq_norm), len(val_trunc_seq_norm), len(test_trunc_seq_norm))
print(seq_trunc_normalizer.keys())

first_vehicle_trunc = next(iter(train_trunc_seq_norm))
print(train_trunc_seq_norm[first_vehicle_trunc].keys())
print(train_trunc_seq_norm[first_vehicle_trunc]["numerical_sequence"].shape)
print(train_trunc_seq_norm[first_vehicle_trunc]["histogram_sequence"].shape)

16485 3532 3533
dict_keys(['means', 'stds'])
dict_keys(['vehicle_id', 'time_steps', 'time_gaps', 'numerical_sequence', 'histogram_sequence', 'sequence_length', 'static_features', 'duration', 'event', 'readout_time', 'original_duration'])
(11, 8)
(11, 97)


In [13]:
# print one truncated sequence
train_trunc_seq_norm[first_vehicle_trunc]

{'vehicle_id': 18034,
 'time_steps': array([ 7.2, 16.6, 18.2, 19.8, 26.2, 37.2, 45. , 53.4, 53.6, 59.2, 65.2]),
 'time_gaps': array([ 0. ,  9.4,  1.6,  1.6,  6.4, 11. ,  7.8,  8.4,  0.2,  5.6,  6. ]),
 'numerical_sequence': array([[-0.83639429, -0.94690403, -0.14421165, -0.32066635, -0.96881126,
         -0.72820438, -0.99799439, -0.5010674 ],
        [-0.68169712, -0.73210283, -0.14421165, -0.32066635, -0.7318738 ,
         -0.70709463, -0.77808051, -0.50080895],
        [-0.65162357, -0.68940925, -0.14421165, -0.32066635, -0.68392322,
         -0.70101943, -0.73385643, -0.50080895],
        [-0.61969296, -0.65525439, -0.14421165, -0.32066635, -0.64569737,
         -0.69666248, -0.6968289 , -0.50055051],
        [-0.50863347, -0.50937776, -0.14421165, -0.32066635, -0.48516435,
         -0.66505921, -0.546438  , -0.49988824],
        [-0.33245717, -0.24303339, -0.14421165, -0.32066635, -0.19471191,
         -0.61762362, -0.28003109, -0.4988383 ],
        [-0.19883968, -0.05888452, -0.1

In [16]:
all_num = np.concatenate(
    [seq["numerical_sequence"] for seq in train_seq_norm.values()],
    axis=0
)

print("NaNs:", np.isnan(all_num).sum())
print("Mean (approx 0):", all_num.mean(axis=0)[:5])
print("Std (approx 1):", all_num.std(axis=0)[:5])

NaNs: 0
Mean (approx 0): [-6.38522869e-17 -1.92144990e-16  2.64971090e-15  1.26473172e-14
 -6.64998667e-16]
Std (approx 1): [1. 1. 1. 1. 1.]


In [31]:
sequence_builder.save_sequence_artifacts(
    train_seq=train_seq_norm,
    val_seq=val_seq_norm,
    test_seq=test_seq_norm,
    normalizer=seq_normalizer,
    output_dir=SEQUENCE_DIR
)

Sequences and normalizer saved to: C:\Users\patri\OneDrive\Documents\SDSBO\Master Thesis\code_ssl_ttf\data\processed\sequences


In [11]:
sequence_builder.save_truncated_sequence_artifacts(
    train_trunc_seq=train_trunc_seq_norm,
    val_trunc_seq=val_trunc_seq_norm,
    test_trunc_seq=test_trunc_seq_norm,
    normalizer_trunc=seq_trunc_normalizer,
    output_dir=SEQUENCE_DIR
)

Truncated sequences and normalizer saved to: C:\Users\patri\OneDrive\Documents\SDSBO\Master Thesis\code_ssl_ttf\data\processed\sequences


## Sequence prefixes for SSL pretraining
To construct the self-supervised pretraining dataset, multiple prefixes were sampled from each full operational trajectory. Each prefix consisted of the sequence of readouts from the beginning of the vehicle history up to a selected intermediate observation point. Prefix endpoints were sampled at fixed proportions of the observed sequence length to capture early, middle, and late operational stages while avoiding excessive redundancy. A minimum prefix length was enforced to ensure sufficient temporal context, and the number of prefixes per vehicle was capped to prevent longer trajectories from dominating pretraining.

Prefixes sampled from full operational trajectories constitute the fundamental training instances for self-supervised pretraining. For each prefix, two stochastic augmented views are generated using modality-specific perturbations. Numerical features are augmented using masking, noise injection, and dropout, while histogram features are perturbed through masking only to preserve their distributional structure. A contrastive learning objective is then applied to enforce representation invariance across augmented views of the same prefix while maintaining discrimination between different prefixes.

For each full vehicle trajectory, up to three strict prefixes whose endpoints correspond to approximately 30%, 60%, and 90% of the sequence length are generated. Prefixes are retained only if they contain at least five observations. Duplicate prefix lengths arising from rounding are removed.